In [0]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 28.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ['KAGGLE_USERNAME']= "YOUR_USER_NAME"
os.environ['KAGGLE_KEY']="YOUR_KAGGLE_KEY"

In [0]:
spark.sql(
    """
    create schema if not exists workspace.codebasics
    """
)

DataFrame[]

In [0]:
spark.sql("""
create volume if not exists workspace.codebasics.customer_spending
""")

DataFrame[]

In [0]:
# Download the Kaggle dataset to the specified volume and unzip
# !kaggle datasets download -d mountboy/online-store-customer-transactions-1m-rows -p /Volumes/workspace/codebasics/customer_spending --unzip

In [0]:
df = spark.read.table("codebasics.customer_spending_1_m")
display(df)

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
722760,2023-02-12T01:40:00.000Z,Female,43,Single,Hawaii,Platinum,Unemployment,PayPal,0,null
722761,2023-02-12T01:47:00.000Z,Female,54,Married,Delaware,Basic,Employees,Other,1,2557.92
722762,2023-02-12T01:51:00.000Z,Male,26,Married,North Dakota,Platinum,Unemployment,Card,null,857.01
722763,2023-02-12T01:53:00.000Z,Female,38,Married,Texas,Basic,Unemployment,PayPal,1,null
722764,2023-02-12T01:54:00.000Z,Female,78,Single,Alaska,Basic,workers,PayPal,0,null
722765,2023-02-12T01:58:00.000Z,Male,65,Single,West Virginia,Missing,workers,Card,0,1840.03
722766,2023-02-12T01:59:00.000Z,Male,66,Single,Nebraska,Basic,workers,PayPal,1,81.37
722767,2023-02-12T02:05:00.000Z,Male,26,Married,Michigan,Silver,self-employed,PayPal,1,2463.77
722768,2023-02-12T02:08:00.000Z,Female,40,Single,Alaska,Platinum,workers,Other,1,null
722769,2023-02-12T02:11:00.000Z,Female,28,Married,Nebraska,Platinum,Unemployment,Card,1,1575.98


### Content
There are 11 features.

- Transaction_date - Transaction date
- Transaction_ID - This is a unique transaction id
- Gender - Customer Gender
- Age - Customer Age
- Marital_status - Marital status about customer
- State_names - Customer location of State.
- Segment - Customer membership
- Employees_status - Customer employment status
- Payment_method - Payment method used by customer
- Referral - Customer coming from referral link or not
- Amount_spent - Amount spent by customer per transaction

In [0]:
df.count(),len(df.columns)

(1000000, 11)

In [0]:
df.columns

['Transaction_ID',
 'Transaction_date',
 'Gender',
 'Age',
 'Marital_status',
 'State_names',
 'Segment',
 'Employees_status',
 'Payment_method',
 'Referral',
 'Amount_spent']

In [0]:
df.printSchema()

root
 |-- Transaction_ID: integer (nullable = true)
 |-- Transaction_date: timestamp (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Marital_status: string (nullable = true)
 |-- State_names: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Employees_status: string (nullable = true)
 |-- Payment_method: string (nullable = true)
 |-- Referral: integer (nullable = true)
 |-- Amount_spent: double (nullable = true)



In [0]:
import pyspark.sql.functions as F

display(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]))

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
0,0,11052,16654,0,0,0,10225,0,61812,96065


In [0]:
display(df.groupBy("State_names").count().orderBy(F.desc("count")))

State_names,count
Illinois,26694
Georgia,25620
Massachusetts,25194
Maine,24491
Minnesota,23820
Kentucky,23799
Missouri,22815
Delaware,22467
Arizona,22220
California,22018


In [0]:
spark.sql("""
          create volume if not exists workspace.codebasics.bronze
""")

DataFrame[]

In [0]:
df.write.format("delta").mode("overwrite").save("/Volumes/workspace/codebasics/bronze/customer_spending")

In [0]:
display(spark.read.load("/Volumes/workspace/codebasics/bronze/customer_spending"))

Transaction_ID,Transaction_date,Gender,Age,Marital_status,State_names,Segment,Employees_status,Payment_method,Referral,Amount_spent
1000,2018-01-01T00:04:00.000Z,Female,39,Single,Oklahoma,Platinum,Unemployment,Card,0,1557.5
1001,2018-01-01T00:06:00.000Z,Male,34,Married,Hawaii,Basic,workers,PayPal,1,153.55
1002,2018-01-01T00:14:00.000Z,Female,53,Married,Iowa,Basic,self-employed,PayPal,1,2640.96
1003,2018-01-01T00:23:00.000Z,Male,33,Married,Wisconsin,Basic,self-employed,Card,1,293.58
1004,2018-01-01T00:25:00.000Z,Female,36,Married,Texas,Platinum,Employees,Card,0,1608.01
1005,2018-01-01T00:27:00.000Z,null,48,Married,North Carolina,Silver,self-employed,Other,null,1001.32
1006,2018-01-01T00:29:00.000Z,Female,55,Single,Ohio,Silver,Unemployment,PayPal,0,1553.61
1007,2018-01-01T00:29:00.000Z,Female,35,Married,Hawaii,Basic,workers,Other,1,1851.58
1008,2018-01-01T00:31:00.000Z,Male,18,Married,Nevada,Basic,Employees,Card,1,null
1009,2018-01-01T00:33:00.000Z,Female,50,Married,North Dakota,Basic,workers,Card,1,541.23
